# GPT-2 Architecture Walkthrough

This notebook illustrates the GPT-2 architecture using the Hugging Face transformers library.
We'll examine each component of the decoder-only transformer without downloading the full model weights.

In [1]:
import torch
import torch.nn as nn
from transformers import GPT2Config, GPT2LMHeadModel
import math

## 1. GPT-2 Configuration

Let's look at the hyperparameters that define GPT-2's architecture.

In [2]:
# Create a GPT-2 config (small version)
config = GPT2Config()

print("GPT-2 Small Configuration:")
print(f"  Vocabulary size: {config.vocab_size:,}")
print(f"  Max sequence length: {config.n_positions:,}")
print(f"  Embedding dimension (d_model): {config.n_embd}")
print(f"  Number of layers: {config.n_layer}")
print(f"  Number of attention heads: {config.n_head}")
print(f"  Head dimension (d_k = d_v): {config.n_embd // config.n_head}")
print(f"  FFN inner dimension: {config.n_inner}")

GPT-2 Small Configuration:
  Vocabulary size: 50,257
  Max sequence length: 1,024
  Embedding dimension (d_model): 768
  Number of layers: 12
  Number of attention heads: 12
  Head dimension (d_k = d_v): 64
  FFN inner dimension: None


## 2. Initialize Model (Random Weights)

We create the model architecture with random weights - no download needed!

In [3]:
# Initialize model with random weights (no pretrained weights downloaded)
model = GPT2LMHeadModel(config)

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,} ({total_params / 1e6:.1f}M)")

Total parameters: 124,439,808 (124.4M)


In [21]:
124439808 - 124439808

0

## 3. Model Architecture Overview

Let's print the full model structure.

In [4]:
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


## 4. Component-by-Component Breakdown

### 4.1 Token and Position Embeddings

In [5]:
# Access the transformer backbone
transformer = model.transformer

# Token embeddings: vocab_size x n_embd
wte = transformer.wte  # Word Token Embeddings
print("Token Embeddings (wte):")
print(f"  Shape: {wte.weight.shape}")
print(f"  Parameters: {wte.weight.numel():,}")

# Position embeddings: n_positions x n_embd  
wpe = transformer.wpe  # Word Position Embeddings
print("\nPosition Embeddings (wpe):")
print(f"  Shape: {wpe.weight.shape}")
print(f"  Parameters: {wpe.weight.numel():,}")

total_embed = wte.weight.numel() + wpe.weight.numel()
print(f"\nTotal embedding parameters: {total_embed:,}")

Token Embeddings (wte):
  Shape: torch.Size([50257, 768])
  Parameters: 38,597,376

Position Embeddings (wpe):
  Shape: torch.Size([1024, 768])
  Parameters: 786,432

Total embedding parameters: 39,383,808


### 4.2 A Single Transformer Block

Each block contains:
1. Layer Norm 1
2. Multi-Head Causal Self-Attention
3. Residual Connection
4. Layer Norm 2
5. Feed-Forward Network (MLP)
6. Residual Connection

In [6]:
# Get the first transformer block
block = transformer.h[0]
print("Transformer Block Structure:")
print(block)

Transformer Block Structure:
GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attn): GPT2Attention(
    (c_attn): Conv1D(nf=2304, nx=768)
    (c_proj): Conv1D(nf=768, nx=768)
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): GPT2MLP(
    (c_fc): Conv1D(nf=3072, nx=768)
    (c_proj): Conv1D(nf=768, nx=3072)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)


### 4.3 Layer Normalization

In [7]:
# Layer norm before attention
ln_1 = block.ln_1
print("Layer Norm 1 (before attention):")
print(f"  Normalized shape: {ln_1.normalized_shape}")
print(f"  Weight (gamma): {ln_1.weight.shape}")
print(f"  Bias (beta): {ln_1.bias.shape}")
print(f"  Parameters: {ln_1.weight.numel() + ln_1.bias.numel():,}")

# Layer norm before FFN
ln_2 = block.ln_2
print("\nLayer Norm 2 (before FFN):")
print(f"  Parameters: {ln_2.weight.numel() + ln_2.bias.numel():,}")

Layer Norm 1 (before attention):
  Normalized shape: (768,)
  Weight (gamma): torch.Size([768])
  Bias (beta): torch.Size([768])
  Parameters: 1,536

Layer Norm 2 (before FFN):
  Parameters: 1,536


### 4.4 Multi-Head Causal Self-Attention

GPT-2 uses a combined QKV projection for efficiency.

In [8]:
attn = block.attn
print("Multi-Head Attention:")
print(f"  Number of heads: {attn.num_heads}")
print(f"  Head dimension: {attn.head_dim}")
print(f"  Embedding dimension: {attn.embed_dim}")

# Combined Q, K, V projection (n_embd -> 3 * n_embd)
c_attn = attn.c_attn
print(f"\nCombined QKV Projection (c_attn):")
print(f"  Weight shape: {c_attn.weight.shape}")
print(f"  This projects to Q, K, V concatenated: {config.n_embd} -> {3 * config.n_embd}")
print(f"  Parameters: {c_attn.weight.numel() + c_attn.bias.numel():,}")

# Output projection (n_embd -> n_embd)
c_proj = attn.c_proj
print(f"\nOutput Projection (c_proj):")
print(f"  Weight shape: {c_proj.weight.shape}")
print(f"  Parameters: {c_proj.weight.numel() + c_proj.bias.numel():,}")

attn_params = (c_attn.weight.numel() + c_attn.bias.numel() + 
               c_proj.weight.numel() + c_proj.bias.numel())
print(f"\nTotal attention parameters per block: {attn_params:,}")

Multi-Head Attention:
  Number of heads: 12
  Head dimension: 64
  Embedding dimension: 768

Combined QKV Projection (c_attn):
  Weight shape: torch.Size([768, 2304])
  This projects to Q, K, V concatenated: 768 -> 2304
  Parameters: 1,771,776

Output Projection (c_proj):
  Weight shape: torch.Size([768, 768])
  Parameters: 590,592

Total attention parameters per block: 2,362,368


In [9]:
d = 768
print(4 * d * d + 4 * d)

2362368


### 4.5 Understanding the QKV Projection

Let's see how the combined projection splits into Q, K, V.

In [ ]:
# The c_attn weight projects input to concatenated [Q, K, V]
# Shape: (n_embd, 3 * n_embd) in GPT-2's Conv1D format

n_embd = config.n_embd
n_head = config.n_head
head_dim = n_embd // n_head

print("QKV Projection Breakdown:")
print(f"  Input dimension: {n_embd}")
print(f"  Output dimension: {3 * n_embd} = 3 x {n_embd}")
print(f"")
print(f"  After projection, split into:")
print(f"    Q: {n_embd} = {n_head} heads x {head_dim} head_dim")
print(f"    K: {n_embd} = {n_head} heads x {head_dim} head_dim")
print(f"    V: {n_embd} = {n_head} heads x {head_dim} head_dim")

# Verify: this matches 4 * d^2 for attention
# QKV: n_embd * 3 * n_embd = 3 * n_embd^2
# Output: n_embd * n_embd = n_embd^2
# Total: 4 * n_embd^2
print(f"")
print(f"Parameter count formula:")
print(f"  QKV projection: {n_embd} x {3 * n_embd} = {n_embd * 3 * n_embd:,}")
print(f"  Output projection: {n_embd} x {n_embd} = {n_embd * n_embd:,}")
print(f"  Total (weights only): {4 * n_embd * n_embd:,} = 4 x d^2")

### 4.6 Causal Mask

GPT-2 uses a causal mask to prevent attending to future tokens.

In [10]:
# GPT-2 registers a causal mask as a buffer
# Let's visualize what the causal mask looks like

seq_len = 8  # Example sequence length

# Create causal mask (lower triangular)
causal_mask = torch.tril(torch.ones(seq_len, seq_len))

print(f"Causal Mask for sequence length {seq_len}:")
print("(1 = can attend, 0 = cannot attend)")
print()
for i in range(seq_len):
    row = [f"{int(causal_mask[i, j])}" for j in range(seq_len)]
    print(f"  Position {i}: [{' '.join(row)}]")

print()
print("Each row shows which positions that token can attend to.")
print("Position 0 can only see itself.")
print("Position 7 can see all previous positions (0-7).")

Causal Mask for sequence length 8:
(1 = can attend, 0 = cannot attend)

  Position 0: [1 0 0 0 0 0 0 0]
  Position 1: [1 1 0 0 0 0 0 0]
  Position 2: [1 1 1 0 0 0 0 0]
  Position 3: [1 1 1 1 0 0 0 0]
  Position 4: [1 1 1 1 1 0 0 0]
  Position 5: [1 1 1 1 1 1 0 0]
  Position 6: [1 1 1 1 1 1 1 0]
  Position 7: [1 1 1 1 1 1 1 1]

Each row shows which positions that token can attend to.
Position 0 can only see itself.
Position 7 can see all previous positions (0-7).


### 4.7 Feed-Forward Network (MLP)

In [11]:
mlp = block.mlp
print("Feed-Forward Network (MLP):")
print(mlp)

# First linear layer: n_embd -> n_inner (expansion)
c_fc = mlp.c_fc
print(f"\nFirst Linear (c_fc) - Expansion:")
print(f"  Weight shape: {c_fc.weight.shape}")
print(f"  {config.n_embd} -> {config.n_inner} (4x expansion)")
print(f"  Parameters: {c_fc.weight.numel() + c_fc.bias.numel():,}")

# GELU activation (no parameters)
print(f"\nActivation: {mlp.act} (no parameters)")

# Second linear layer: n_inner -> n_embd (projection)
c_proj = mlp.c_proj
print(f"\nSecond Linear (c_proj) - Projection:")
print(f"  Weight shape: {c_proj.weight.shape}")
print(f"  {config.n_inner} -> {config.n_embd}")
print(f"  Parameters: {c_proj.weight.numel() + c_proj.bias.numel():,}")

mlp_params = (c_fc.weight.numel() + c_fc.bias.numel() + 
              c_proj.weight.numel() + c_proj.bias.numel())
print(f"\nTotal MLP parameters per block: {mlp_params:,}")

Feed-Forward Network (MLP):
GPT2MLP(
  (c_fc): Conv1D(nf=3072, nx=768)
  (c_proj): Conv1D(nf=768, nx=3072)
  (act): NewGELUActivation()
  (dropout): Dropout(p=0.1, inplace=False)
)

First Linear (c_fc) - Expansion:
  Weight shape: torch.Size([768, 3072])
  768 -> None (4x expansion)
  Parameters: 2,362,368

Activation: NewGELUActivation() (no parameters)

Second Linear (c_proj) - Projection:
  Weight shape: torch.Size([3072, 768])
  None -> 768
  Parameters: 2,360,064

Total MLP parameters per block: 4,722,432


### 4.8 Final Layer Norm and LM Head

In [12]:
# Final layer norm after all transformer blocks
ln_f = transformer.ln_f
print("Final Layer Norm (ln_f):")
print(f"  Parameters: {ln_f.weight.numel() + ln_f.bias.numel():,}")

# Language modeling head: projects to vocabulary
lm_head = model.lm_head
print(f"\nLanguage Model Head (lm_head):")
print(f"  Weight shape: {lm_head.weight.shape}")
print(f"  Projects {config.n_embd} -> {config.vocab_size} (vocabulary)")

# Note: GPT-2 ties the LM head weights with token embeddings
print(f"\nWeight Tying:")
print(f"  lm_head.weight is wte.weight: {lm_head.weight is transformer.wte.weight}")
print(f"  This saves {config.vocab_size * config.n_embd:,} parameters!")

Final Layer Norm (ln_f):
  Parameters: 1,536

Language Model Head (lm_head):
  Weight shape: torch.Size([50257, 768])
  Projects 768 -> 50257 (vocabulary)

Weight Tying:
  lm_head.weight is wte.weight: True
  This saves 38,597,376 parameters!


In [20]:
d = 768
V = 50257
L = 1024
total = V * d + L * d + (12 * d * d + 13 * d) * 12 + 2 * d
print(total)

124439808


## 5. Complete Parameter Count

In [ ]:
def count_parameters(model, config):
    """Count parameters by component."""
    
    n_embd = config.n_embd
    n_layer = config.n_layer
    n_inner = config.n_inner
    if n_inner is None:
        n_inner = n_embd * 4
    vocab_size = config.vocab_size
    n_positions = config.n_positions
    
    counts = {}
    
    # Token embeddings
    counts['token_embeddings'] = vocab_size * n_embd
    
    # Position embeddings
    counts['position_embeddings'] = n_positions * n_embd
    
    # Per block
    # Layer norms: 2 per block, each has weight and bias of size n_embd
    ln_per_block = 2 * (n_embd + n_embd)  # 2 * (weight + bias)
    
    # Attention: QKV projection + output projection (with biases)
    attn_per_block = (n_embd * 3 * n_embd + 3 * n_embd +  # c_attn weight + bias
                      n_embd * n_embd + n_embd)            # c_proj weight + bias
    
    # MLP: expansion + projection (with biases)
    mlp_per_block = (n_embd * n_inner + n_inner +    # c_fc weight + bias
                     n_inner * n_embd + n_embd)      # c_proj weight + bias
    
    counts['layer_norms_per_block'] = ln_per_block
    counts['attention_per_block'] = attn_per_block
    counts['mlp_per_block'] = mlp_per_block
    counts['total_per_block'] = ln_per_block + attn_per_block + mlp_per_block
    
    # All blocks
    counts['all_blocks'] = n_layer * counts['total_per_block']
    
    # Final layer norm
    counts['final_layer_norm'] = n_embd + n_embd
    
    # LM head (tied with token embeddings, so not counted separately)
    counts['lm_head'] = 0  # Weight tied
    
    # Total
    counts['total'] = (counts['token_embeddings'] + 
                       counts['position_embeddings'] + 
                       counts['all_blocks'] + 
                       counts['final_layer_norm'])
    
    return counts

counts = count_parameters(model, config)

print("=" * 60)
print("GPT-2 Small Parameter Count")
print("=" * 60)
print(f"\nEmbeddings:")
print(f"  Token embeddings:    {counts['token_embeddings']:>12,}  ({config.vocab_size} x {config.n_embd})")
print(f"  Position embeddings: {counts['position_embeddings']:>12,}  ({config.n_positions} x {config.n_embd})")

print(f"\nPer Transformer Block:")
print(f"  Layer norms (2x):    {counts['layer_norms_per_block']:>12,}  (2 x 2 x {config.n_embd})")
print(f"  Attention:           {counts['attention_per_block']:>12,}  (~4 x d^2)")
print(f"  MLP:                 {counts['mlp_per_block']:>12,}  (~2 x d x d_ff)")
print(f"  Block total:         {counts['total_per_block']:>12,}")

print(f"\nAll {config.n_layer} Blocks:         {counts['all_blocks']:>12,}")

print(f"\nFinal layer norm:      {counts['final_layer_norm']:>12,}")
print(f"LM head (tied):        {counts['lm_head']:>12,}  (shares token embedding weights)")

print(f"\n" + "=" * 60)
print(f"TOTAL:                 {counts['total']:>12,}  ({counts['total']/1e6:.1f}M)")
print("=" * 60)

# Verify against actual model
actual = sum(p.numel() for p in model.parameters())
print(f"\nActual model parameters: {actual:,}")
print(f"Our calculation:         {counts['total']:,}")
print(f"Match: {actual == counts['total']}")

## 6. Comparing GPT-2 Sizes

In [ ]:
# Different GPT-2 configurations
gpt2_configs = {
    'GPT-2 Small': {'n_layer': 12, 'n_embd': 768, 'n_head': 12},
    'GPT-2 Medium': {'n_layer': 24, 'n_embd': 1024, 'n_head': 16},
    'GPT-2 Large': {'n_layer': 36, 'n_embd': 1280, 'n_head': 20},
    'GPT-2 XL': {'n_layer': 48, 'n_embd': 1600, 'n_head': 25},
}

print(f"{'Model':<15} {'Layers':>8} {'d_model':>8} {'Heads':>8} {'d_ff':>8} {'Params':>12}")
print("-" * 65)

for name, params in gpt2_configs.items():
    cfg = GPT2Config(
        vocab_size=50257,
        n_positions=1024,
        n_layer=params['n_layer'],
        n_embd=params['n_embd'],
        n_head=params['n_head'],
        n_inner=4 * params['n_embd'],
    )
    m = GPT2LMHeadModel(cfg)
    total = sum(p.numel() for p in m.parameters())
    
    print(f"{name:<15} {params['n_layer']:>8} {params['n_embd']:>8} {params['n_head']:>8} {4*params['n_embd']:>8} {total:>12,}")